In [1]:
#234567890#234567890#234567890#234567890#234567890#234567890#234567890#23456789
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
BATCH = 64

In [34]:
DATA = './data'

### Working with Data

In [3]:
training_data = datasets.FashionMNIST(
    root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.FashionMNIST(
    root='data', train=False, download=True, transform=ToTensor())

In [4]:
train_dataloader = DataLoader(training_data, batch_size=BATCH)
test_dataloader = DataLoader(test_data, batch_size=BATCH)

for X, y in test_dataloader:
    print(f'X.shape: {X.shape} (n, c, h, w)')
    print(f'y.shape: {y.shape} ({y.dtype})')
    break

X.shape: torch.Size([64, 1, 28, 28]) (n, c, h, w)
y.shape: torch.Size([64]) (torch.int64)


### Creating Models

In [5]:
device = (
    torch.accelerator.current_accelerator().type 
    if torch.accelerator.is_available() 
    else 'cpu')
print('Device:', device)

Device: mps


In [22]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10))

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [23]:
mod = NeuralNetwork().to(device)
print(mod)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [24]:
loss_f = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(mod.parameters(), lr=1e-3)

In [28]:
def train(loader, mod, loss_f, opt):
    size = len(loader.dataset)
    mod.train()
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        preds = mod(X)
        loss = loss_f(preds, y)
        loss.backward()
        opt.step()
        opt.zero_grad()
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f'loss: {loss:>7f} ({current:>5d}/{size:>5d})')

In [29]:
def test(loader, mod, loss_f):
    size = len(loader.dataset)
    n_batches = len(loader)
    mod.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = mod(X)
            test_loss += loss_f(preds, y).item()
            correct += (preds.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= n_batches
    correct /= size
    print(
        f'Test Err:\nAcc: {(100 * correct):>0.2f}%, '
        f'Avg. loss: {test_loss:>8.8f}\n')

In [30]:
epochs = 5
for t in range(epochs):
    print(f'Epoch: {t + 1}')
    print('-' * 35)
    train(test_dataloader, mod, loss_f, optimizer)
    test(test_dataloader, mod, loss_f)
print('Done!')

Epoch: 1
-----------------------------------
loss: 2.307415 (   64/10000)
loss: 2.290067 ( 6464/10000)
Test Err:
Acc: 10.34%, Avg. loss: 2.28675801

Epoch: 2
-----------------------------------
loss: 2.283510 (   64/10000)
loss: 2.265501 ( 6464/10000)
Test Err:
Acc: 11.14%, Avg. loss: 2.26348365

Epoch: 3
-----------------------------------
loss: 2.260885 (   64/10000)
loss: 2.241352 ( 6464/10000)
Test Err:
Acc: 16.75%, Avg. loss: 2.24033449

Epoch: 4
-----------------------------------
loss: 2.238273 (   64/10000)
loss: 2.216889 ( 6464/10000)
Test Err:
Acc: 30.59%, Avg. loss: 2.21643706

Epoch: 5
-----------------------------------
loss: 2.214831 (   64/10000)
loss: 2.191025 ( 6464/10000)
Test Err:
Acc: 41.53%, Avg. loss: 2.19090984

Done!


In [41]:
path = f'{DATA}/mod.pth'
torch.save(mod.state_dict(), path)
print('Model saved to', path)

Model saved to ./data/mod.pth


In [42]:
mod = NeuralNetwork().to(device)
mod.load_state_dict(torch.load(path, weights_only=True))

<All keys matched successfully>

In [44]:
classes = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt',
    'Sneaker', 'Bag', 'Ankle boot']
mod.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    preds = mod(x)
    pred, actual = classes[preds[0].argmax(0)], classes[y]
    print(f'Predicted: "{pred}", Actual: "{actual}"')

Predicted: "Sandal", Actual: "Ankle boot"
